# Test Conversation Variables

Este notebook permite probar y validar la lógica de `create_conv_variables` antes de aplicar cambios al pipeline.

## Flujo:
1. Leer tabla `bt_conv_conversations_bck` de Geospot
2. Replicar/modificar la función `create_conv_variables`
3. Probar y validar resultados
4. Cuando esté listo, aplicar al pipeline


In [1]:
# Imports y configuración
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine

# Cargar credenciales desde .env local (en la misma carpeta que este notebook)
# override=True fuerza sobrescribir variables de entorno existentes
env_path = Path("/Users/cesaracosta/Documents/SPOT2/dagster/dagster-pipeline/src/dagster_pipeline/defs/conversation_analysis/notebooks/.env")

if env_path.exists():
    # override=True es crucial para sobrescribir valores None o vacíos del kernel
    load_dotenv(env_path, override=True)
    print(f"✅ Cargado .env desde: {env_path}")
else:
    print(f"❌ No existe: {env_path}")

# Configuración Geospot desde .env
GEOSPOT_HOST = os.getenv("SQL_SERVER_HOST_geo")
GEOSPOT_USER = os.getenv("SQL_SERVER_USER_geo")
GEOSPOT_PASSWORD = os.getenv("SQL_SERVER_PASSWORD_geo")
GEOSPOT_DB = os.getenv("SQL_SERVER_DB_geo")
GEOSPOT_PORT = os.getenv("SQL_SERVER_PORT_geo", "5432")
TABLE_NAME = "bt_conv_conversations"

print(f"✅ Imports cargados")
print(f"📊 Host: {GEOSPOT_HOST}")
print(f"📊 User: {GEOSPOT_USER}")
print(f"📊 Database: {GEOSPOT_DB}")
print(f"📊 Port: {GEOSPOT_PORT}")


✅ Cargado .env desde: /Users/cesaracosta/Documents/SPOT2/dagster/dagster-pipeline/src/dagster_pipeline/defs/conversation_analysis/notebooks/.env
✅ Imports cargados
📊 Host: gdtoxqaj1v5lvq.cqt3kzfdcy2o.us-east-1.rds.amazonaws.com
📊 User: cesar.acosta
📊 Database: geospot
📊 Port: 5432


In [ ]:
# Conectar a Geospot y leer la tabla
engine_url = (
    f"postgresql+psycopg2://{GEOSPOT_USER}:{GEOSPOT_PASSWORD}"
    f"@{GEOSPOT_HOST}:{GEOSPOT_PORT}/{GEOSPOT_DB}"
)
engine = create_engine(engine_url)

print(f"📥 Leyendo tabla {TABLE_NAME}...")
with engine.connect() as conn:
    df = pd.read_sql(f"SELECT * FROM bt_conv_conversations_bck", conn)

print(f"✅ Leídos {len(df)} registros")
print(f"📊 Columnas: {list(df.columns)}")
df.head()


📥 Leyendo tabla bt_conv_conversations...
✅ Leídos 5923 registros
📊 Columnas: ['conv_id', 'lead_phone_number', 'conv_start_date', 'conv_end_date', 'conv_text', 'conv_messages', 'conv_last_message_date', 'lead_id', 'lead_created_at', 'conv_variables', 'vis_pseudo_id', 'spot_id', 'aud_inserted_date', 'aud_inserted_at', 'aud_updated_date', 'aud_updated_at', 'aud_job']


,conv_id,lead_phone_number,conv_start_date,conv_end_date,conv_text,conv_messages,conv_last_message_date,lead_id,lead_created_at,conv_variables,vis_pseudo_id,spot_id,aud_inserted_date,aud_inserted_at,aud_updated_date,aud_updated_at,aud_job
0,13235251029_20251221,13235251029,2025-12-21 12:37:02,2025-12-21 13:25:22,"[2025-12-21 12:37:02] user: ¡Hola, me interesa...","[{'id': 163128, 'type': 'user', 'message_body'...",2025-12-21 13:25:22,60607,2025-12-21 12:37:04,"[{'conv_variable_name': 'messages_count', 'con...",None,None,2026-01-08,2026-01-08 12:37:08.820220,2026-01-08,2026-01-08 12:37:08.820220,conversation_analysis_pipeline
1,14053803489_20260104,14053803489,2026-01-04 08:34:37,2026-01-04 08:34:43,[2026-01-04 08:34:37] user: Solo 2 pasos: ¡Úne...,"[{'id': 165654, 'type': 'user', 'message_body'...",2026-01-04 08:34:43,60127,2025-12-09 23:20:24,"[{'conv_variable_name': 'messages_count', 'con...",None,None,2026-01-08,2026-01-08 12:37:08.820220,2026-01-08,2026-01-08 12:37:08.820220,conversation_analysis_pipeline
2,15403539080_20251217,15403539080,2025-12-17 18:24:27,2025-12-18 11:50:32,[2025-12-17 18:24:27] user: \n[2025-12-17 18:2...,"[{'id': 161998, 'type': 'user', 'message_body'...",2025-12-18 11:50:32,60295,2025-12-14 08:10:04,"[{'conv_variable_name': 'messages_count', 'con...",None,None,2026-01-08,2026-01-08 12:37:08.820220,2026-01-08,2026-01-08 12:37:08.820220,conversation_analysis_pipeline
3,16154034371_20260107,16154034371,2026-01-07 14:34:35,2026-01-07 14:38:59,"[2026-01-07 14:34:35] user: ¡Hola, me interesa...","[{'id': 167598, 'type': 'user', 'message_body'...",2026-01-07 14:38:59,61168,2026-01-07 14:34:36,"[{'conv_variable_name': 'messages_count', 'con...",None,None,2026-01-08,2026-01-08 12:37:08.820220,2026-01-08,2026-01-08 12:37:08.820220,conversation_analysis_pipeline
4,523315175862_20251224,523315175862,2025-12-24 09:22:42,2025-12-24 09:34:02,"[2025-12-24 09:22:42] user: ¡Hola, me interesa...","[{'id': 163671, 'type': 'user', 'message_body'...",2025-12-24 09:34:02,60680,2025-12-24 09:22:43,"[{'conv_variable_name': 'messages_count', 'con...",None,None,2026-01-08,2026-01-08 12:37:08.820220,2026-01-08,2026-01-08 12:37:08.820220,conversation_analysis_pipeline


## Función create_conv_variables (RÉPLICA DEL PIPELINE)

Esta función replica exactamente la lógica actual del pipeline (`processors.py`).
Adaptada a los nombres de columnas de la tabla `bt_conv_conversations_bck`.


In [ ]:
def create_conv_variables_current(row: pd.Series) -> list:
    """
    RÉPLICA EXACTA de la función del pipeline (processors.py).
    Adaptada a columnas de bt_conv_conversations_bck:
    - conv_messages (en vez de messages)
    - conv_text (en vez de conversation_text)
    - conv_start_date (en vez of conversation_start)
    - conv_end_date (en vez of conversation_end)
    """
    variables = []
    
    # Parse messages from JSON string
    messages = row.get('conv_messages', [])
    if isinstance(messages, str):
        try:
            messages = json.loads(messages)
        except:
            messages = []
    if not isinstance(messages, list):
        messages = []
    
    # 1. Messages count
    variables.append({
        'conv_variable_category': 'metric',
        'conv_variable_name': 'messages_count',
        'conv_variable_value': len(messages)
    })
    
    # 2. User messages count
    user_messages = [m for m in messages if isinstance(m, dict) and m.get("type") == "user"]
    variables.append({
        'conv_variable_category': 'metric',
        'conv_variable_name': 'user_messages_count',
        'conv_variable_value': len(user_messages)
    })
    
    # 3. Total conversation time (hours)
    start = row.get("conv_start_date")
    end = row.get("conv_end_date")
    total_time = None
    if pd.notnull(start) and pd.notnull(end):
        try:
            start_ts = pd.to_datetime(start)
            end_ts = pd.to_datetime(end)
            total_time = (end_ts - start_ts).total_seconds() / 3600
        except:
            pass
    variables.append({
        'conv_variable_category': 'metric',
        'conv_variable_name': 'total_time_conversation',
        'conv_variable_value': total_time
    })
    
    # 4. User average response time (minutes)
    response_times = []
    prev_assistant_time = None
    for m in messages:
        if not isinstance(m, dict):
            continue
        m_type = m.get('type')
        m_time = m.get('created_at_utc') or m.get('created_at')
        try:
            m_time = pd.to_datetime(m_time)
        except:
            m_time = None
        if m_type == 'assistant' and m_time is not None:
            prev_assistant_time = m_time
        elif m_type == 'user' and m_time is not None and prev_assistant_time is not None:
            delta = (m_time - prev_assistant_time).total_seconds() / 60.0
            if delta >= 0:
                response_times.append(delta)
            prev_assistant_time = None
    
    avg_response = np.mean(response_times) if len(response_times) > 0 else None
    variables.append({
        'conv_variable_category': 'metric',
        'conv_variable_name': 'user_average_time_respond_minutes',
        'conv_variable_value': avg_response
    })
    
    # 5. Visit intention tag (LÓGICA ACTUAL DEL PIPELINE)
    visit_intent_phrases = [
        'Podemos solicitar una visita a partir de',
        'Indícanos qué día y hora te conviene y lo gestionamos',
        'Por favor, indícame qué día y hora te conviene para solicitar la visita',
        'Estoy aquí para ayudarte a coordinarla',
        'He encontrado esta fecha válida:'
    ]
    conversation_text = str(row.get('conv_text', ''))
    has_intent = any(phrase in conversation_text for phrase in visit_intent_phrases)
    variables.append({
        'conv_variable_category': 'tag',
        'conv_variable_name': 'visit_intention',
        'conv_variable_value': 1 if has_intent else 0
    })
    
    return variables

print("✅ Función create_conv_variables_current definida (réplica del pipeline)")


✅ Función create_conv_variables_current definida (réplica del pipeline)


## Función MODIFICADA para probar

Aquí puedes modificar la lógica de `visit_intention` y probar cambios.
Esta versión incluye la nueva lógica: detectar si el usuario dice "agendar visita" o "agendar cita".


In [5]:
def create_conv_variables_new(row: pd.Series) -> list:
    """
    VERSIÓN MODIFICADA para probar cambios.
    
    CAMBIO EN visit_intention:
    - ANTES: Solo detecta frases del asistente
    - AHORA: También detecta si el usuario dice "agendar visita" o "agendar cita"
    """
    variables = []
    
    # Parse messages from JSON string
    messages = row.get('conv_messages', [])
    if isinstance(messages, str):
        try:
            messages = json.loads(messages)
        except:
            messages = []
    if not isinstance(messages, list):
        messages = []
    
    # 1. Messages count
    variables.append({
        'conv_variable_category': 'metric',
        'conv_variable_name': 'messages_count',
        'conv_variable_value': len(messages)
    })
    
    # 2. User messages count
    user_messages = [m for m in messages if isinstance(m, dict) and m.get("type") == "user"]
    variables.append({
        'conv_variable_category': 'metric',
        'conv_variable_name': 'user_messages_count',
        'conv_variable_value': len(user_messages)
    })
    
    # 3. Total conversation time (hours)
    start = row.get("conv_start_date")
    end = row.get("conv_end_date")
    total_time = None
    if pd.notnull(start) and pd.notnull(end):
        try:
            start_ts = pd.to_datetime(start)
            end_ts = pd.to_datetime(end)
            total_time = (end_ts - start_ts).total_seconds() / 3600
        except:
            pass
    variables.append({
        'conv_variable_category': 'metric',
        'conv_variable_name': 'total_time_conversation',
        'conv_variable_value': total_time
    })
    
    # 4. User average response time (minutes)
    response_times = []
    prev_assistant_time = None
    for m in messages:
        if not isinstance(m, dict):
            continue
        m_type = m.get('type')
        m_time = m.get('created_at_utc') or m.get('created_at')
        try:
            m_time = pd.to_datetime(m_time)
        except:
            m_time = None
        if m_type == 'assistant' and m_time is not None:
            prev_assistant_time = m_time
        elif m_type == 'user' and m_time is not None and prev_assistant_time is not None:
            delta = (m_time - prev_assistant_time).total_seconds() / 60.0
            if delta >= 0:
                response_times.append(delta)
            prev_assistant_time = None
    
    avg_response = np.mean(response_times) if len(response_times) > 0 else None
    variables.append({
        'conv_variable_category': 'metric',
        'conv_variable_name': 'user_average_time_respond_minutes',
        'conv_variable_value': avg_response
    })
    
    # 5. Visit intention tag (LÓGICA MODIFICADA)
    # ========================================
    # A) Frases del ASISTENTE (lógica original)
    assistant_intent_phrases = [
        'Podemos solicitar una visita a partir de',
        'Indícanos qué día y hora te conviene y lo gestionamos',
        'Por favor, indícame qué día y hora te conviene para solicitar la visita',
        'Estoy aquí para ayudarte a coordinarla',
        'He encontrado esta fecha válida:'
    ]
    conversation_text = str(row.get('conv_text', ''))
    has_assistant_intent = any(phrase in conversation_text for phrase in assistant_intent_phrases)
    
    # B) NUEVO: Frases del USUARIO (case insensitive)
    user_intent_phrases = ['agendar visita', 'agendar cita']
    has_user_intent = False
    for msg in messages:
        if isinstance(msg, dict) and msg.get('type') == 'user':
            msg_body = str(msg.get('message_body', '')).lower()
            if any(phrase in msg_body for phrase in user_intent_phrases):
                has_user_intent = True
                break
    
    # visit_intention = 1 si cualquiera de los dos
    has_intent = has_assistant_intent or has_user_intent
    variables.append({
        'conv_variable_category': 'tag',
        'conv_variable_name': 'visit_intention',
        'conv_variable_value': 1 if has_intent else 0
    })
    
    return variables

print("✅ Función create_conv_variables_new definida (con nueva lógica de visit_intention)")


✅ Función create_conv_variables_new definida (con nueva lógica de visit_intention)


## Aplicar y comparar resultados


In [6]:
# Función auxiliar para extraer visit_intention de las variables
def extract_visit_intention(conv_vars) -> int:
    """Extrae el valor de visit_intention de la lista de variables."""
    if isinstance(conv_vars, str):
        try:
            conv_vars = json.loads(conv_vars)
        except:
            return None
    if isinstance(conv_vars, list):
        for v in conv_vars:
            if isinstance(v, dict) and v.get('conv_variable_name') == 'visit_intention':
                return v.get('conv_variable_value')
    return None

print("✅ Función extract_visit_intention definida")


✅ Función extract_visit_intention definida


In [7]:
# Aplicar ambas funciones a cada fila
print("🔄 Aplicando funciones a todas las filas...")

# Aplicar función ACTUAL (réplica del pipeline)
df['vars_current'] = df.apply(create_conv_variables_current, axis=1)

# Aplicar función NUEVA (con lógica modificada)
df['vars_new'] = df.apply(create_conv_variables_new, axis=1)

# Extraer visit_intention de ambas
df['visit_intention_current'] = df['vars_current'].apply(extract_visit_intention)
df['visit_intention_new'] = df['vars_new'].apply(extract_visit_intention)

# Extraer visit_intention de la tabla original (lo que ya está en producción)
df['visit_intention_prod'] = df['conv_variables'].apply(extract_visit_intention)

print(f"✅ Procesadas {len(df)} filas")


🔄 Aplicando funciones a todas las filas...
✅ Procesadas 5923 filas


In [8]:
# Comparación de resultados
print("=" * 60)
print("📊 COMPARACIÓN DE VISIT_INTENTION")
print("=" * 60)

print(f"\n🔵 PRODUCCIÓN ACTUAL (datos en tabla):")
print(f"   visit_intention = 1: {(df['visit_intention_prod'] == 1).sum()}")
print(f"   visit_intention = 0: {(df['visit_intention_prod'] == 0).sum()}")
print(f"   visit_intention = None: {df['visit_intention_prod'].isna().sum()}")

print(f"\n🟢 RÉPLICA PIPELINE (create_conv_variables_current):")
print(f"   visit_intention = 1: {(df['visit_intention_current'] == 1).sum()}")
print(f"   visit_intention = 0: {(df['visit_intention_current'] == 0).sum()}")

print(f"\n🟡 NUEVA LÓGICA (create_conv_variables_new):")
print(f"   visit_intention = 1: {(df['visit_intention_new'] == 1).sum()}")
print(f"   visit_intention = 0: {(df['visit_intention_new'] == 0).sum()}")

# Diferencias
nuevos_detectados = (df['visit_intention_new'] == 1) & (df['visit_intention_current'] == 0)
print(f"\n🆕 NUEVOS DETECTADOS con lógica modificada: {nuevos_detectados.sum()}")
print("   (Conversaciones que pasarían de 0 a 1 con el cambio)")


📊 COMPARACIÓN DE VISIT_INTENTION

🔵 PRODUCCIÓN ACTUAL (datos en tabla):
   visit_intention = 1: 630
   visit_intention = 0: 5293
   visit_intention = None: 0

🟢 RÉPLICA PIPELINE (create_conv_variables_current):
   visit_intention = 1: 630
   visit_intention = 0: 5293

🟡 NUEVA LÓGICA (create_conv_variables_new):
   visit_intention = 1: 1356
   visit_intention = 0: 4567

🆕 NUEVOS DETECTADOS con lógica modificada: 726
   (Conversaciones que pasarían de 0 a 1 con el cambio)


In [9]:
# Ver ejemplos de conversaciones NUEVAS detectadas
df_nuevos = df[nuevos_detectados][['conv_id', 'conv_text', 'conv_messages']].head(10)

print("📋 EJEMPLOS de conversaciones NUEVAS con visit_intention=1:")
print("=" * 80)

for i, (idx, row) in enumerate(df_nuevos.iterrows()):
    print(f"\n🆔 [{i+1}] {row['conv_id']}")
    print("-" * 60)
    
    # Buscar el mensaje del usuario que activó la detección
    messages = row['conv_messages']
    if isinstance(messages, str):
        try:
            messages = json.loads(messages)
        except:
            messages = []
    
    # Mostrar mensajes del usuario que contienen "agendar"
    for msg in messages:
        if isinstance(msg, dict) and msg.get('type') == 'user':
            msg_body = str(msg.get('message_body', '')).lower()
            if 'agendar visita' in msg_body or 'agendar cita' in msg_body:
                print(f"   👤 Usuario: \"{msg.get('message_body')}\"")
    
    print("-" * 60)


📋 EJEMPLOS de conversaciones NUEVAS con visit_intention=1:

🆔 [1] 523315175862_20251224
------------------------------------------------------------
   👤 Usuario: "Agendar visita"
------------------------------------------------------------

🆔 [2] 18177164744_20260103
------------------------------------------------------------
   👤 Usuario: "Agendar visita"
------------------------------------------------------------

🆔 [3] 51949177725_20251027
------------------------------------------------------------
   👤 Usuario: "agendar visita"
------------------------------------------------------------

🆔 [4] 51949177726_20251027
------------------------------------------------------------
   👤 Usuario: "agendar visita"
------------------------------------------------------------

🆔 [5] 51949177734_20251104
------------------------------------------------------------
   👤 Usuario: "agendar visita"
------------------------------------------------------------

🆔 [6] 51949177744_20251110
-------

In [15]:
df_nuevos

,conv_id,conv_text,conv_messages
4,523315175862_20251224,"[2025-12-24 09:22:42] user: ¡Hola, me interesa...","[{'id': 163671, 'type': 'user', 'message_body'..."
5,18177164744_20260103,"[2026-01-03 16:06:23] user: ¡Hola, me interesa...","[{'id': 165533, 'type': 'user', 'message_body'..."
10,51949177725_20251027,"[2025-10-27 11:21:05] user: ¡Hola, me interesa...","[{'id': 140563, 'type': 'user', 'message_body'..."
11,51949177726_20251027,"[2025-10-27 11:37:36] user: ¡Hola, me interesa...","[{'id': 140577, 'type': 'user', 'message_body'..."
18,51949177734_20251104,"[2025-11-04 06:12:47] user: ¡Hola, me interesa...","[{'id': 144068, 'type': 'user', 'message_body'..."
23,51949177744_20251110,[2025-11-10 05:26:14] user: hola\n[2025-11-10 ...,"[{'id': 146447, 'type': 'user', 'message_body'..."
52,51949177779_20251202,"[2025-12-02 17:45:44] user: ¡Hola, me interesa...","[{'id': 156010, 'type': 'user', 'message_body'..."
54,522214293626_20260107,"[2026-01-07 10:25:36] user: ¡Hola, me interesa...","[{'id': 167315, 'type': 'user', 'message_body'..."
64,5213318213182_20251023,"[2025-10-23 10:14:59] user: ¡Hola, me interesa...","[{'id': 138319, 'type': 'user', 'message_body'..."
68,522227075068_20260106,"[2026-01-06 17:38:27] user: ¡Hola, me interesa...","[{'id': 167148, 'type': 'user', 'message_body'..."


In [16]:
df_nuevos[df_nuevos['conv_id'] == '51949177725_20251027']

,conv_id,conv_text,conv_messages
10,51949177725_20251027,"[2025-10-27 11:21:05] user: ¡Hola, me interesa...","[{'id': 140563, 'type': 'user', 'message_body'..."


In [12]:
# Búsqueda directa: todos los mensajes de usuario con "agendar visita" o "agendar cita"
print("🔍 BÚSQUEDA: Mensajes de usuario con 'agendar visita' o 'agendar cita'")
print("=" * 80)

user_intent_phrases = ['agendar visita', 'agendar cita']
found_messages = []

for idx, row in df.iterrows():
    messages = row.get('conv_messages', [])
    if isinstance(messages, str):
        try:
            messages = json.loads(messages)
        except:
            continue
    
    for msg in messages:
        if isinstance(msg, dict) and msg.get('type') == 'user':
            msg_body = str(msg.get('message_body', '')).lower()
            for phrase in user_intent_phrases:
                if phrase in msg_body:
                    found_messages.append({
                        'conv_id': row['conv_id'],
                        'phrase': phrase,
                        'message': msg.get('message_body')
                    })

print(f"\n📊 Total mensajes encontrados: {len(found_messages)}")
print(f"📊 Conversaciones únicas: {len(set(m['conv_id'] for m in found_messages))}")

# Mostrar primeros 15
print("\n📋 Primeros 15 mensajes encontrados:")
for i, m in enumerate(found_messages[:15]):
    print(f"   [{i+1}] {m['conv_id']}: \"{m['message']}\"")


🔍 BÚSQUEDA: Mensajes de usuario con 'agendar visita' o 'agendar cita'

📊 Total mensajes encontrados: 1102
📊 Conversaciones únicas: 958

📋 Primeros 15 mensajes encontrados:
   [1] 523315175862_20251224: "Agendar visita"
   [2] 18177164744_20260103: "Agendar visita"
   [3] 51949177725_20251027: "agendar visita"
   [4] 51949177726_20251027: "agendar visita"
   [5] 51949177734_20251104: "agendar visita"
   [6] 51949177743_20251107: "agendar visita"
   [7] 51949177744_20251110: "agendar visita"
   [8] 51949177753_20251112: "agendar visita"
   [9] 51949177779_20251202: "agendar visita"
   [10] 51949177781_20251202: "agendar visita"
   [11] 51949177781_20251202: "agendar visita"
   [12] 522214293626_20260107: "Agendar visita"
   [13] 51949177782_20251203: "agendar visita"
   [14] 51949177782_20251203: "agendar visita"
   [15] 5213318213182_20251023: "Agendar Visita"
